# Carrinho IESB — agente de rotas

Adaptação do gabarito do Laboratório 9 (Aula 10) para a planta do térreo enviada em 12/11/2026. Sem simulação. O mapa é um **grafo simplificado**, documentado em `MAPA_DO_GRAFO.md`.

Cada sala, banheiro e local nomeado é um nó. O Pátio/Direito tem Mesa 1 a Mesa 4. Cada ligação vale **1 trecho**, pois a planta não traz distâncias de cada porta e mesa. Logo, o custo não é em metros. O LLM escolhe a ferramenta; o A* calcula a rota.


In [ ]:
%pip -q install transformers


## 1. Mapa do IESB

As ligações abaixo são bidirecionais. Corredores e pontos do eixo também são nós. O desenho escreve “D-02” dentro do Bloco C; aqui usamos “C-02” como correção provisória da sequência.


In [ ]:
LOCAIS = [
    "Entrada Principal", "Eixo - Norte", "Eixo - Design", "Eixo - Bloco C",
    "Eixo - Bloco B", "Eixo - Fundo",
    "Pátio/Direito", "Mesa 1", "Mesa 2", "Mesa 3", "Mesa 4",
    "Sala dos Professores", "Gastro", "Oasis Smart",
    "WC Acessível Pátio", "WC Feminino Pátio",
    "Cinema", "WC Cinema", "Anexo Creche", "Biblioteca",
    "Design", "Corredor Design", "D-01", "D-02", "D-03", "D-04", "D-05", "D-06",
    "WC Masculino Design", "WC Feminino Design",
    "Bloco C", "Corredor Bloco C", "C-01", "C-02", "C-03", "C-04", "C-05", "C-06",
    "WC Masculino C", "WC Feminino C",
    "Bloco B", "Corredor Bloco B", "B-01", "B-02", "B-03", "B-04", "B-05", "B-06",
    "WC Masculino B", "WC Feminino B",
    "Acesso Lateral I/J", "Corredor I/J", "Bloco I", "Bloco J",
    "I-01", "I-02", "I-03", "I-04", "I-05", "I-06",
    "J-01", "J-02", "J-03", "J-04", "J-05", "J-06",
    "WC Masculino I", "WC Feminino I", "WC Masculino J", "WC Feminino J",
]

LIGACOES = [
    ("Entrada Principal", "Eixo - Norte"),
    ("Eixo - Norte", "Eixo - Design"),
    ("Eixo - Design", "Eixo - Bloco C"),
    ("Eixo - Bloco C", "Eixo - Bloco B"),
    ("Eixo - Bloco B", "Eixo - Fundo"),
    ("Eixo - Norte", "Pátio/Direito"),
    ("Eixo - Norte", "Cinema"),
    ("Eixo - Norte", "Biblioteca"),
    ("Eixo - Design", "Corredor Design"),
    ("Eixo - Bloco C", "Corredor Bloco C"),
    ("Eixo - Bloco B", "Corredor Bloco B"),
    ("Eixo - Bloco C", "Acesso Lateral I/J"),
    ("Acesso Lateral I/J", "Corredor I/J"),
    ("Corredor Design", "Design"),
    ("Corredor Bloco C", "Bloco C"),
    ("Corredor Bloco B", "Bloco B"),
    ("Corredor I/J", "Bloco I"),
    ("Corredor I/J", "Bloco J"),
]

for local in ["Mesa 1", "Mesa 2", "Mesa 3", "Mesa 4", "Sala dos Professores",
              "Gastro", "Oasis Smart", "WC Acessível Pátio", "WC Feminino Pátio"]:
    LIGACOES.append(("Pátio/Direito", local))

for local in ["WC Cinema", "Anexo Creche"]:
    LIGACOES.append(("Cinema", local))

for local in ["D-01", "D-02", "D-03", "D-04", "D-05", "D-06",
              "WC Masculino Design", "WC Feminino Design"]:
    LIGACOES.append(("Corredor Design", local))

for local in ["C-01", "C-02", "C-03", "C-04", "C-05", "C-06",
              "WC Masculino C", "WC Feminino C"]:
    LIGACOES.append(("Corredor Bloco C", local))

for local in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06",
              "WC Masculino B", "WC Feminino B"]:
    LIGACOES.append(("Corredor Bloco B", local))

for local in ["I-01", "I-02", "I-03", "I-04", "I-05", "I-06",
              "J-01", "J-02", "J-03", "J-04", "J-05", "J-06",
              "WC Masculino I", "WC Feminino I", "WC Masculino J", "WC Feminino J"]:
    LIGACOES.append(("Corredor I/J", local))

VIZINHOS = {}
for local in LOCAIS:
    VIZINHOS[local] = []
for origem, destino in LIGACOES:
    VIZINHOS[origem].append((destino, 1))
    VIZINHOS[destino].append((origem, 1))

print(len(LOCAIS), "nós e", len(LIGACOES), "ligações bidirecionais")


## 2. A* do gabarito adaptado

A heurística vale zero porque não há coordenadas confiáveis para todos os nós. Assim, a função mantém o A* do gabarito e encontra o menor número de trechos definido neste grafo.


In [ ]:
def h(local, destino):
    return 0


def a_estrela(origem, destino):
    fronteira = [(h(origem, destino), 0, origem, [origem])]
    visitados = []
    while len(fronteira) > 0:
        melhor = min(fronteira)
        fronteira.remove(melhor)
        f, g, local, caminho = melhor
        if local == destino:
            return caminho, g
        if local in visitados:
            continue
        visitados.append(local)
        for vizinho, custo in VIZINHOS[local]:
            if vizinho not in visitados:
                novo_g = g + custo
                fronteira.append((novo_g + h(vizinho, destino), novo_g,
                                  vizinho, caminho + [vizinho]))
    return None, None


print(a_estrela("Entrada Principal", "Mesa 4"))


## 3. Ferramentas que o modelo pode pedir

Como no gabarito, cada ferramenta é uma função com descrição e parâmetros. O código confere os nomes antes de calcular.


In [ ]:
def calcular_rota(origem: str, destino: str) -> str:
    """
    Calcula com A* a rota com menos trechos entre dois locais da planta do IESB.

    Args:
        origem: local de partida, com nome igual ao da lista de locais
        destino: local de chegada, com nome igual ao da lista de locais
    """
    if origem not in LOCAIS:
        return "Erro: o local " + str(origem) + " não existe no mapa."
    if destino not in LOCAIS:
        return "Erro: o local " + str(destino) + " não existe no mapa."
    caminho, custo = a_estrela(origem, destino)
    if caminho is None:
        return "Não existe caminho entre " + origem + " e " + destino + "."
    return "Caminho: " + " -> ".join(caminho) + ". Custo total: " + str(custo) + " trechos."


def listar_vizinhos(local: str) -> str:
    """
    Lista os locais ligados diretamente a um local da planta do IESB.

    Args:
        local: local consultado, com nome igual ao da lista de locais
    """
    if local not in LOCAIS:
        return "Erro: o local " + str(local) + " não existe no mapa."
    partes = []
    for vizinho, custo in VIZINHOS[local]:
        partes.append(vizinho)
    return "Vizinhos de " + local + ": " + ", ".join(partes) + "."


FERRAMENTAS = [calcular_rota, listar_vizinhos]
print(calcular_rota("Entrada Principal", "Mesa 4"))
print(listar_vizinhos("Pátio/Direito"))


## 4. Carregar o Qwen (mesmo modelo do gabarito)

No Colab, escolha GPU T4 se estiver disponível. O mesmo modelo público do gabarito é carregado pelo `transformers`.


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    dispositivo = "cuda"
else:
    dispositivo = "cpu"
print("Dispositivo:", dispositivo)

nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)
if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()


def conversar(mensagens, ferramentas=None, max_tokens=200):
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, tools=ferramentas, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    return tokenizador_qwen.decode(saida[0][tamanho_da_pergunta:], skip_special_tokens=True)


print("Qwen carregado.")


## 5. Agente com ferramentas

O modelo decide se precisa de uma ferramenta e informa os argumentos. O Python executa a função pedida e devolve o resultado ao modelo. O modelo responde em português usando esse resultado.


In [ ]:
REGRAS = (
    "Você ajuda a encontrar rotas no térreo do IESB. "
    "Para perguntas sobre caminhos, custos e locais vizinhos, use as ferramentas. "
    "Nunca invente rotas, distâncias em metros ou acessos: use apenas o resultado das ferramentas. "
    "O custo é contado em trechos, não em metros. "
    "Responda em português, de forma curta. "
    "Nomes válidos dos locais: " + ", ".join(LOCAIS) + "."
)


def extrair_json(texto):
    inicio = texto.find("{")
    fim = texto.rfind("}")
    if inicio == -1 or fim == -1:
        return None
    try:
        return json.loads(texto[inicio:fim + 1])
    except:
        return None


def ler_pedido_de_ferramenta(resposta):
    if "<tool_call>" not in resposta:
        return None
    return extrair_json(resposta)


def executar(pedido):
    nome = pedido.get("name")
    argumentos = pedido.get("arguments", {})
    if nome == "calcular_rota":
        return calcular_rota(argumentos.get("origem"), argumentos.get("destino"))
    if nome == "ferramenta_do_trio":
        return ferramenta_do_trio(argumentos.get("origem"), argumentos.get("destino"))
    if nome == "listar_vizinhos":
        return listar_vizinhos(argumentos.get("local"))
    return "Erro: a ferramenta " + str(nome) + " não existe."


def agente(pergunta, mostrar=True):
    mensagens = [
        {"role": "system", "content": REGRAS},
        {"role": "user", "content": pergunta},
    ]
    for passo in range(3):
        resposta = conversar(mensagens, FERRAMENTAS)
        pedido = ler_pedido_de_ferramenta(resposta)
        if pedido is None:
            return resposta
        resultado = executar(pedido)
        if mostrar:
            print("   [ferramenta]", pedido.get("name"), pedido.get("arguments"))
            print("   [resultado ]", resultado)
        mensagens.append({"role": "assistant", "content": resposta})
        mensagens.append({"role": "tool", "content": resultado})
    return "Não consegui concluir em 3 passos."


## 6. Perguntas para experimentar

O primeiro teste é curto; o segundo atravessa blocos; o terceiro não pede cálculo. As respostas do LLM dependem da execução do Qwen.


In [ ]:
perguntas = [
    "Como vou da Entrada Principal até a Mesa 2?",
    "Qual é a rota de B-04 para I-05?",
    "Oi! O que você faz?",
]
for pergunta in perguntas:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()


## 7. Ferramenta do trio (desafio final do gabarito)

Aqui os estados do trio são os locais do IESB. A função abaixo usa o mesmo A* da célula 2 e devolve texto para o modelo. Ela corresponde à `ferramenta_do_trio` do gabarito do professor.


In [ ]:
def a_estrela_trio(origem, destino):
    return a_estrela(origem, destino)


ESTADOS_DO_TRIO = LOCAIS


def ferramenta_do_trio(origem: str, destino: str) -> str:
    """
    Calcula a rota com menos trechos entre dois locais do térreo do IESB usando A*.

    Args:
        origem: local de partida, escrito como na lista de locais do IESB
        destino: local de chegada, escrito como na lista de locais do IESB
    """
    if origem not in ESTADOS_DO_TRIO or destino not in ESTADOS_DO_TRIO:
        return "Erro: local fora da lista. Locais válidos: " + ", ".join(ESTADOS_DO_TRIO)
    caminho, custo = a_estrela_trio(origem, destino)
    if caminho is None:
        return "Não existe caminho entre " + origem + " e " + destino + "."
    return "Caminho: " + " -> ".join(caminho) + ". Custo total: " + str(custo) + " trechos."


print(ferramenta_do_trio("Entrada Principal", "Mesa 1"))


### O que o modelo enxerga

Como no gabarito, `get_json_schema` mostra o nome, a descrição e os argumentos da ferramenta que são enviados ao Qwen. O modelo não lê o corpo da função.


In [ ]:
from transformers.utils import get_json_schema

print(json.dumps(get_json_schema(ferramenta_do_trio), indent=1, ensure_ascii=False))


## 8. Agente do trio

O agente continua com o mesmo ciclo de até três passos do gabarito. Agora a ferramenta de rota é `ferramenta_do_trio`; `listar_vizinhos` continua disponível para perguntas sobre ligações diretas.


In [ ]:
FERRAMENTAS = [ferramenta_do_trio, listar_vizinhos]

perguntas_do_trio = [
    "Como vou da Entrada Principal até a Mesa 1?",
    "Qual o caminho de B-04 para I-05?",
    "Oi! O que você sabe fazer?",
]
for pergunta in perguntas_do_trio:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", agente(pergunta))
    print()


## 9. Testar a escolha da ferramenta

Estes casos seguem a Etapa 5 do gabarito. A contagem mede **a primeira decisão do Qwen**, não a correção do caminho calculado pelo A*.


In [ ]:
CASOS = [
    ("Como vou da Entrada Principal até a Mesa 2?", "ferramenta_do_trio"),
    ("Qual é a rota de B-04 para I-05?", "ferramenta_do_trio"),
    ("Quais locais estão ligados ao Pátio/Direito?", "listar_vizinhos"),
    ("O que há ao lado do Corredor Design?", "listar_vizinhos"),
    ("Bom dia!", None),
    ("Obrigado pela ajuda!", None),
]

acertos = 0
for pergunta, esperada in CASOS:
    mensagens = [{"role": "system", "content": REGRAS},
                 {"role": "user", "content": pergunta}]
    pedido = ler_pedido_de_ferramenta(conversar(mensagens, FERRAMENTAS))
    if pedido is None:
        escolhida = None
    else:
        escolhida = pedido.get("name")
    if escolhida == esperada:
        acertos = acertos + 1
        marca = "certo "
    else:
        marca = "ERRADO"
    print(marca, "|", pergunta, "| esperada:", esperada, "| escolhida:", escolhida)

print("Escolheu a ferramenta certa em", acertos, "de", len(CASOS), "casos.")


## Conferência

O grafo, o A* e as ferramentas Python podem ser conferidos sem carregar o modelo. As células do Qwen exigem `torch`, `transformers` e o download de `Qwen/Qwen2.5-1.5B-Instruct`. As respostas do modelo e a pontuação dos seis casos só podem ser avaliadas depois dessa execução.

O LLM escolhe a ferramenta e seus argumentos. O A* calcula o menor número de trechos no grafo. Nenhum custo representa metros ou tempo.
